# Silver — Limpeza e Upsert

Este notebook constrói a camada **silver**: lê as tabelas da bronze (`growdev.bronze`), **limpa** os dados (conversão de tipos, padronização de categorias) e grava por **upsert** (`MERGE`) nas tabelas `growdev.silver.<tabela>`, usando a chave primária de cada uma. No caminho, recapitulamos Python essencial (variáveis, listas, dicionários, funções, f-strings) e a API de DataFrames do PySpark (read, select, filter, withColumn, groupBy, join, write).

In [0]:
# Recapitulando: variáveis, f-strings, listas e dicionários

CATALOG = "growdev"
SCHEMA_BRONZE = "bronze"
SCHEMA_SILVER = "silver"

# Dicionário com a chave primária de cada tabela (base do upsert)
chaves = {
    "account_balances": "balance_id",
    "accounts": "account_id",
    "customers": "customer_id",
    "loans": "loan_id",
}

print(f"Origem: {CATALOG}.{SCHEMA_BRONZE} | Destino: {CATALOG}.{SCHEMA_SILVER}")
print("Tabelas:", list(chaves.keys()))

In [0]:
# Recapitulando: funções + withColumn (limpeza de dados)
from pyspark.sql import functions as F

def para_data(nome_coluna):
    """Converte texto em data, aceitando os formatos que aparecem nos dados.

    try_to_date devolve NULL quando o formato não bate (em vez de dar erro),
    e F.coalesce fica com o primeiro formato que funcionar.
    """
    c = F.col(nome_coluna)
    return F.coalesce(
        F.try_to_date(c, "yyyy-MM-dd HH:mm:ss"),
        F.try_to_date(c, "yyyy-MM-dd"),
        F.try_to_date(c, "dd/MM/yyyy"),
        F.try_to_date(c, "dd-MM-yyyy"),
    )

def para_data_hora(nome_coluna):
    """Converte texto em timestamp, aceitando os formatos que aparecem nos dados.

    Aqui usamos F.expr para escrever a função SQL try_to_timestamp com formato,
    que ainda não está disponível direto no PySpark desta versão.
    """
    return F.coalesce(
        F.expr(f"try_to_timestamp({nome_coluna}, 'yyyy-MM-dd HH:mm:ss')"),
        F.expr(f"try_to_timestamp({nome_coluna}, 'yyyy-MM-dd')"),
        F.expr(f"try_to_timestamp({nome_coluna}, 'dd/MM/yyyy')"),
        F.expr(f"try_to_timestamp({nome_coluna}, 'dd-MM-yyyy')"),
    )

def para_valor(nome_coluna):
    """Converte texto de dinheiro ('R$ 1.234,56' ou '1234.56') em número decimal.

    Valores inválidos, como 'N/A', viram NULL em vez de dar erro.
    """
    c = F.col(nome_coluna)
    texto = F.regexp_replace(c, "[R$\\s]", "")  # remove 'R$', '$' e espaços
    formato_br = F.regexp_replace(F.regexp_replace(texto, "\\.", ""), ",", ".")
    limpo = F.when(c.contains(","), formato_br).otherwise(texto)  # vírgula = formato BR
    # só converte se sobrou um número de verdade: 'N/A' não passa no rlike
    numero = F.when(limpo.rlike("^\\d+(\\.\\d+)?$"), limpo).otherwise(None)
    return numero.cast("decimal(14,2)")

# Recapitulando: dicionário de funções — cada tabela tem a sua limpeza
def padronizar_categoria(nome_coluna):
    """Padroniza categorias: 'CHEQUE_ESPECIAL' e 'Cheque Especial' viram o mesmo valor.

    Passos: minúsculas, '_' vira espaço, tira acentos e inicial maiúscula em cada palavra.
    """
    c = F.lower(F.trim(nome_coluna))
    c = F.regexp_replace(c, "_", " ")
    c = F.translate(c, "áàâãéêíóôõúç", "aaaaeeiooouc")
    return F.initcap(c)

def limpar_account_balances(df):
    return df.withColumn("balance_date", para_data("balance_date"))

def limpar_accounts(df):
    return (
        df.withColumn("opened_at", para_data("opened_at"))
        .withColumn("status", F.initcap(F.lower(F.trim("status"))))  # 'ATIVO' -> 'Ativo'
    )

# Recapitulando: dicionários que padronizam os campos de clientes

# Nome do estado -> sigla (siglas que já estão certas, como 'SP', não mudam)
UF = {
    "Acre": "AC", "Alagoas": "AL", "Amapá": "AP", "Amazonas": "AM",
    "Bahia": "BA", "Ceará": "CE", "Distrito Federal": "DF", "Espírito Santo": "ES",
    "Goiás": "GO", "Maranhão": "MA", "Mato Grosso": "MT", "Mato Grosso do Sul": "MS",
    "Minas Gerais": "MG", "Pará": "PA", "Paraíba": "PB", "Paraná": "PR",
    "Pernambuco": "PE", "Piauí": "PI", "Rio de Janeiro": "RJ", "Rio Grande do Norte": "RN",
    "Rio Grande do Sul": "RS", "Rondônia": "RO", "Roraima": "RR", "Santa Catarina": "SC",
    "São Paulo": "SP", "Sergipe": "SE", "Tocantins": "TO",
}

# Variação do segmento (após a padronização) -> nome correto
SEGMENTOS = {
    "Pessoa Fisica": "Pessoa Física",
    "Pessoa Juridica": "Pessoa Jurídica",
    "Pf": "Pessoa Física",
    "Pj": "Pessoa Jurídica",
}

def limpar_customers(df):
    df = (
        df.withColumn("birth_date", para_data("birth_date"))
        .withColumn("signup_date", para_data("signup_date"))
        .withColumn("declared_monthly_income", para_valor("declared_monthly_income"))
        .withColumn("customer_segment", padronizar_categoria("customer_segment"))  # 'PF' -> 'Pf'
        .withColumn("risk_profile", padronizar_categoria("risk_profile"))  # 'BAIXO'/'baixo' -> 'Baixo'
    )
    # df.replace troca os valores usando os dicionários: 'São Paulo' -> 'SP', 'Pf' -> 'Pessoa Física'
    return (
        df.replace(UF, subset=["state"])  # ex.: 'São Paulo' -> 'SP'
        .replace(SEGMENTOS, subset=["customer_segment"])  # ex.: 'Pf' -> 'Pessoa Física'
    )

# Recapitulando: dicionário que unifica sinônimos de categoria
# A padronização acima já deixa tudo em um formato só; aqui juntamos os sinônimos
SINONIMOS = {
    "Antecipacao De Recebiveis": "Antecipacao Recebiveis",
    "Capital De Giro": "Capital Giro",
    "Cap Giro": "Capital Giro",
    "Credito Estudantil": "Estudantil",
    "Credito Imobiliario": "Imobiliario",
    "Emprestimo Pessoal": "Pessoal",
    "Financ Veiculo": "Veiculo",
    "Cred Consignado": "Consignado",
    "Credito Reforma": "Reforma",
}

def limpar_loans(df):
    df = (
        df.withColumn("loan_amount", para_valor("loan_amount"))
        .withColumn("originated_at", para_data_hora("originated_at"))
        .withColumn("status", F.initcap(F.lower(F.trim("status"))))  # 'INADIMPLENTE' -> 'Inadimplente'
        .withColumn("is_default", F.lower(F.trim("is_default")).isin("sim", "s", "true", "1"))  # 'sim'/'TRUE'/'1' -> True
        .withColumn("loan_category", padronizar_categoria("loan_category"))  # 'CHEQUE_ESPECIAL' -> 'Cheque Especial'
    )
    # df.replace troca os valores usando o dicionário de sinônimos
    return df.replace(SINONIMOS, subset=["loan_category"])

limpezas = {
    "account_balances": limpar_account_balances,
    "accounts": limpar_accounts,
    "customers": limpar_customers,
    "loans": limpar_loans,
}
print("Funções de limpeza prontas para:", list(limpezas.keys()))

In [0]:
# A camada silver grava com UPSERT (MERGE): atualiza o que existe e insere o que é novo
def upsert_silver(nome_tabela):
    """Limpa os dados da bronze e grava na silver com MERGE (upsert).

    - Limpeza: aplica a função de limpeza da tabela e marca _processed_at
      (quando o dado foi processado pela silver)
    - Upsert: se a chave primária já existe na silver, ATUALIZA a linha;
      se não existe, INSERE uma nova. É o "atualiza ou insere" do MERGE.
    """
    chave = chaves[nome_tabela]
    destino = f"{CATALOG}.{SCHEMA_SILVER}.{nome_tabela}"  # ex.: growdev.silver.customers

    # 1. Lê a bronze e aplica a limpeza da tabela
    df_bronze = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.{nome_tabela}")
    df_pronto = limpezas[nome_tabela](df_bronze).withColumn(
        "_processed_at", F.current_timestamp()
    )

    # 2. A bronze é append-only: se o mesmo dado chegou em lotes diferentes,
    #    deixa só uma linha por chave primária
    df_pronto = df_pronto.dropDuplicates([chave])

    # 3. Primeira execução cria a tabela; as seguintes fazem MERGE (upsert)
    if not spark.catalog.tableExists(destino):
        df_pronto.write.saveAsTable(destino)
        return destino, "tabela criada"

    df_pronto.createOrReplaceTempView("vw_fonte_upsert")
    spark.sql(
        f"""
        MERGE INTO {destino} AS t
        USING vw_fonte_upsert AS s
          ON t.{chave} = s.{chave}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
        """
    )
    return destino, "upsert (merge)"

In [0]:
# Recapitulando: laço for sobre um dicionário + f-strings
for nome_tabela in chaves:
    destino, acao = upsert_silver(nome_tabela)
    print(f"OK: {nome_tabela} -> {destino} ({acao})")

In [0]:
# Recapitulando: read (spark.table), filter, groupBy, join e agg

# 1) Contagem e chaves duplicadas por tabela (upsert garante 1 linha por chave)
for nome_tabela, chave in chaves.items():
    df = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.{nome_tabela}")
    total = df.count()
    duplicadas = df.groupBy(chave).count().filter(F.col("count") > 1).count()
    print(f"{nome_tabela}: {total} linhas | chaves duplicadas: {duplicadas}")

# 2) Qualidade da limpeza: quantos valores ficaram nulos após a conversão em loans
df_loans = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.loans")
for coluna in ["loan_amount", "originated_at", "is_default"]:
    nulos = df_loans.filter(F.col(coluna).isNull()).count()
    print(f"loans.{coluna}: {nulos} nulos após a limpeza")

# 3) Exemplo com join + groupBy: empréstimos quitados por categoria
resumo_quitados = (
    spark.table(f"{CATALOG}.{SCHEMA_SILVER}.loans")
    .join(spark.table(f"{CATALOG}.{SCHEMA_SILVER}.customers"), "customer_id")
    .filter(F.col("status") == "Quitado")
    .groupBy("loan_category")
    .agg(F.count("*").alias("qtd_emprestimos"), F.avg("loan_amount").alias("valor_medio"))
    .orderBy(F.desc("qtd_emprestimos"))
)
display(resumo_quitados)